main.py


In [ ]:
'''Application entry point and graph initialization for Trip Planner.'''

from __future__ import annotations

from typing import Optional

from langchain_core.messages import AIMessage, HumanMessage
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.graph.state import CompiledStateGraph

from src.config import settings
from src.graph.graph import build_graph

_graph: Optional[CompiledStateGraph] = None


def get_graph() -> CompiledStateGraph:
    '''Initialize the NVIDIA chat model and build the Trip Planner graph once.

    Returns:
        CompiledStateGraph: The cached compiled LangGraph workflow.

    Raises:
        ValueError: If ``API_KEY`` is missing or empty.
    '''
    global _graph
    if _graph is not None:
        return _graph

    if not settings.NVIDIA_API_KEY:
        raise ValueError("api key is not set. Please configure your api_key in your .env")

    llm = ChatNVIDIA(
        model=settings.LLM_MODEL,
        api_key=settings.NVIDIA_API_KEY,
    )
    _graph = build_graph(model=llm)
    return _graph


if __name__ == "__main__":
    graph = get_graph()

    # Persistent conversation history so the agent remembers prior turns.
    messages = []

    while True:
        try:
            _input = input("You: ")
        except (EOFError, KeyboardInterrupt):
            print()
            break

        if _input.strip().lower() in ["exit", "quit"]:
            break

        if not _input.strip():
            continue

        messages.append(HumanMessage(content=_input))

        result = graph.invoke({"messages": messages})

        # Keep the full updated history (includes tool calls + tool results).
        messages = result["messages"]

        final_response = next(
            (
                m
                for m in reversed(messages)
                if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None)
            ),
            None,
        )

        if final_response is not None:
            print("Agent: ", final_response.content)
        else:
            print("Agent:  (no response)")


app.py

In [ ]:
'''Streamlit chat UI for the AI Trip Planner.'''

from __future__ import annotations

import logging
import os

import streamlit as st
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

from main import get_graph
from src.config import settings

logger = logging.getLogger(__name__)

st.set_page_config(page_title="AI Trip Planner", page_icon="✈️", layout="wide")


@st.cache_resource
def load_graph():
    '''Cache the compiled LangGraph workflow across the Streamlit server session.'''
    return get_graph()


def _key_status(env_var: str, settings_val: str | None) -> str:
    '''Return a non-sensitive configuration status label for an API key.'''
    return "✅ Configured" if (os.environ.get(env_var) or settings_val) else "⚠️ Missing"


# Initialize conversation history
if "messages" not in st.session_state:
    st.session_state["messages"] = []

# Sidebar
with st.sidebar:
    st.title("✈️ AI Trip Planner")
    st.caption("Plan trips with AI recommendations and live travel data.")

    st.subheader("Capabilities")
    st.markdown(
        "- **Attractions** — Discover nearby landmarks and points of interest\n"
        "- **Weather** — Check current weather conditions at destinations\n"
        "- **Currency** — Convert amounts using live exchange rates\n"
        "- **Expense Organizer** — Group estimated costs by category\n"
        "- **Expense Total** — Calculate total estimated trip cost"
    )

    st.subheader("API Status")
    st.markdown(
        f"- `NVIDIA_API_KEY`: {_key_status('NVIDIA_API_KEY', settings.NVIDIA_API_KEY)}\n"
        f"- `OPENTRIPMAP_API_KEY`: {_key_status('OPENTRIPMAP_API_KEY', settings.OPENTRIPMAP_API_KEY)}\n"
        f"- `OPENWEATHERMAP_API_KEY`: {_key_status('OPENWEATHERMAP_API_KEY', settings.OPENWEATHERMAP_API_KEY)}"
    )

    if st.button("Clear chat", use_container_width=True):
        st.session_state["messages"] = []
        st.rerun()

# Main chat area
st.title("AI Trip Planner")

if not st.session_state["messages"]:
    with st.chat_message("assistant"):
        st.markdown(
            "Welcome! Tell me where you'd like to go, your travel preferences, "
            "or ask me about attractions, current weather, currency conversions, "
            "and trip expenses."
        )
else:
    for msg in st.session_state["messages"]:
        if isinstance(msg, (SystemMessage, ToolMessage)):
            continue
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            continue
        if isinstance(msg, HumanMessage):
            with st.chat_message("user"):
                st.markdown(msg.content)
        elif isinstance(msg, AIMessage):
            with st.chat_message("assistant"):
                st.markdown(msg.content)

if prompt := st.chat_input("Ask me anything about your trip..."):
    st.session_state["messages"].append(HumanMessage(content=prompt))
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        try:
            graph = load_graph()
        except ValueError as err:
            st.error(str(err))
        except Exception:
            logger.exception("Failed to initialize Trip Planner graph.")
            st.error("Something went wrong. Please try again.")
        else:
            try:
                with st.spinner("Planning your trip..."):
                    result = graph.invoke({"messages": st.session_state["messages"]})
                response = next(
                    (
                        m
                        for m in reversed(result["messages"])
                        if isinstance(m, AIMessage) and not getattr(m, "tool_calls", None)
                    ),
                    None,
                )
                st.session_state["messages"] = result["messages"]
                if response is not None:
                    st.markdown(response.content)
                st.rerun()
            except Exception:
                logger.exception("Error during Trip Planner graph invocation.")
                st.error("Something went wrong. Please try again.")


### answers from GPT

main.py

In [ ]:
"""Application entry point and graph initialization for Trip Planner."""

from __future__ import annotations

from typing import Optional, Sequence

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.graph.state import CompiledStateGraph

from src.config import settings
from src.graph.graph import build_graph

_graph: Optional[CompiledStateGraph] = None


def get_graph() -> CompiledStateGraph:
    """Initialize the NVIDIA chat model and build the Trip Planner graph once."""
    global _graph
    if _graph is not None:
        return _graph

    if not settings.NVIDIA_API_KEY:
        raise ValueError("api key is not set. Please configure your api_key in your .env")

    llm = ChatNVIDIA(
        model=settings.LLM_MODEL,
        api_key=settings.NVIDIA_API_KEY,
    )
    _graph = build_graph(model=llm)
    return _graph


def run_turn(
    graph: CompiledStateGraph,
    history: Sequence[BaseMessage],
    user_input: str,
) -> tuple[list[BaseMessage], AIMessage | None]:
    """Run one conversation turn and return the complete updated history.

    The full message list (including assistant tool calls and tool results) is
    passed to LangGraph and retained for the next turn. Callers should keep the
    returned history in their own session/conversation state.
    """
    messages = [*history, HumanMessage(content=user_input)]
    result = graph.invoke({"messages": messages})
    updated_messages = list(result["messages"])

    final_response = next(
        (
            message
            for message in reversed(updated_messages)
            if isinstance(message, AIMessage)
            and not getattr(message, "tool_calls", None)
        ),
        None,
    )
    return updated_messages, final_response


if __name__ == "__main__":
    graph = get_graph()

    # CLI conversation history lives for this process/session only.
    messages: list[BaseMessage] = []

    while True:
        try:
            user_input = input("You: ")
        except (EOFError, KeyboardInterrupt):
            print()
            break

        if user_input.strip().lower() in {"exit", "quit"}:
            break

        if not user_input.strip():
            continue

        messages, final_response = run_turn(graph, messages, user_input)

        if final_response is not None:
            print("Agent:", final_response.content)
        else:
            print("Agent: (no response)")


app.py

In [ ]:
"""Streamlit chat UI for the AI Trip Planner."""

from __future__ import annotations

import logging
import os

import streamlit as st
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage

from main import get_graph, run_turn
from src.config import settings

logger = logging.getLogger(__name__)

st.set_page_config(page_title="AI Trip Planner", page_icon="✈️", layout="wide")


@st.cache_resource
def load_graph():
    """Cache the compiled LangGraph workflow across Streamlit reruns."""
    return get_graph()


def _key_status(env_var: str, settings_val: str | None) -> str:
    """Return a non-sensitive configuration status label for an API key."""
    return "✅ Configured" if (os.environ.get(env_var) or settings_val) else "⚠️ Missing"


# Streamlit reruns the script after interactions. session_state preserves this
# conversation's complete LangChain message history across those reruns.
if "messages" not in st.session_state:
    st.session_state["messages"] = []

# Sidebar
with st.sidebar:
    st.title("✈️ AI Trip Planner")
    st.caption("Plan trips with AI recommendations and live travel data.")

    st.subheader("Capabilities")
    st.markdown(
        "- **Attractions** — Discover nearby landmarks and points of interest\n"
        "- **Weather** — Check current weather conditions at destinations\n"
        "- **Currency** — Convert amounts using live exchange rates\n"
        "- **Expense Organizer** — Group estimated costs by category\n"
        "- **Expense Total** — Calculate total estimated trip cost"
    )

    st.subheader("API Status")
    st.markdown(
        f"- `NVIDIA_API_KEY`: {_key_status('NVIDIA_API_KEY', settings.NVIDIA_API_KEY)}\n"
        f"- `OPENTRIPMAP_API_KEY`: {_key_status('OPENTRIPMAP_API_KEY', settings.OPENTRIPMAP_API_KEY)}\n"
        f"- `OPENWEATHERMAP_API_KEY`: {_key_status('OPENWEATHERMAP_API_KEY', settings.OPENWEATHERMAP_API_KEY)}"
    )

    if st.button("Clear chat", use_container_width=True):
        st.session_state["messages"] = []
        st.rerun()

# Main chat area
st.title("AI Trip Planner")

if not st.session_state["messages"]:
    with st.chat_message("assistant"):
        st.markdown(
            "Welcome! Tell me where you'd like to go, your travel preferences, "
            "or ask me about attractions, current weather, currency conversions, "
            "and trip expenses."
        )
else:
    for msg in st.session_state["messages"]:
        if isinstance(msg, (SystemMessage, ToolMessage)):
            continue
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            continue
        if isinstance(msg, HumanMessage):
            with st.chat_message("user"):
                st.markdown(msg.content)
        elif isinstance(msg, AIMessage):
            with st.chat_message("assistant"):
                st.markdown(msg.content)

if prompt := st.chat_input("Ask me anything about your trip..."):
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        try:
            graph = load_graph()
            with st.spinner("Planning your trip..."):
                updated_messages, response = run_turn(
                    graph,
                    st.session_state["messages"],
                    prompt,
                )

            # Save the full graph state, not just the visible assistant text.
            # This preserves prior turns plus tool calls/tool results.
            st.session_state["messages"] = updated_messages

            if response is not None:
                st.markdown(response.content)
            else:
                st.markdown("I couldn't produce a response. Please try again.")

        except ValueError as err:
            logger.warning("Trip Planner configuration error: %s", err)
            st.error(str(err))
        except Exception:
            logger.exception("Error during Trip Planner graph invocation.")
            st.error("Something went wrong. Please try again.")
